# AWPO Training on Qwen3-4B

Adaptive Integration of Reasoning Rewards (arxiv 2512.19126) —
mixed outcome + reasoning advantage with variance-aware gating.


## Installation


In [ ]:
import os
os.environ["UNSLOTH_VLLM_STANDBY"] = "1"

try:
    import unsloth
except ImportError:
    !pip install unsloth vllm
try:
    import trl
except ImportError:
    !pip install trl


## Load Model with Unsloth


In [ ]:
from unsloth import FastLanguageModel
import torch

max_seq_length = 2048
lora_rank = 32

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Qwen3-4B-Base",
    max_seq_length=max_seq_length,
    load_in_4bit=False,
    fast_inference=True,
    max_lora_rank=lora_rank,
    gpu_memory_utilization=0.9,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=lora_rank,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_alpha=lora_rank * 2,
    use_gradient_checkpointing="unsloth",
    random_state=3407,
)


## Chat Template Setup


In [ ]:
reasoning_start = "<start_working_out>"
reasoning_end = "<end_working_out>"
solution_start = "<SOLUTION>"
solution_end = "</SOLUTION>"

system_prompt = f"""You are given a problem.
Think about the problem and provide your working out.
Place it between {reasoning_start} and {reasoning_end}.
Then, provide your solution between {solution_start}{solution_end}"""

chat_template = (
    "{% if messages[0]['role'] == 'system' %}"
        "{{ messages[0]['content'] + eos_token }}"
        "{% set loop_messages = messages[1:] %}"
    "{% else %}"
        "{{'{system_prompt}' + eos_token }}"
        "{% set loop_messages = messages %}"
    "{% endif %}"
    "{% for message in loop_messages %}"
        "{% if message['role'] == 'user' %}"
            "{{ message['content'] }}"
        "{% elif message['role'] == 'assistant' %}"
            "{{ message['content'] + eos_token }}"
        "{% endif %}"
    "{% endfor %}"
    "{% if add_generation_prompt %}{{'{reasoning_start}'}}"
    "{% endif %}"
)
chat_template = (
    chat_template
    .replace("'{system_prompt}'", f"'{system_prompt}'")
    .replace("'{reasoning_start}'", f"'{reasoning_start}'")
)
tokenizer.chat_template = chat_template


## Pre Fine-Tuning for Formatting


In [ ]:
from datasets import load_dataset
import pandas as pd
import numpy as np

dataset = load_dataset("unsloth/OpenMathReasoning-mini", split="cot")
dataset = dataset.to_pandas()[["expected_answer", "problem", "generated_solution"]]
is_number = pd.to_numeric(pd.Series(dataset["expected_answer"]), errors="coerce").notnull()
dataset = dataset.iloc[np.where(is_number)[0]]

def format_dataset(x):
    expected_answer = x["expected_answer"]
    problem = x["problem"]
    thoughts = x["generated_solution"].replace("<think>", "").replace("</think>", "").strip()
    final_prompt = (
        reasoning_start + thoughts + reasoning_end +
        solution_start + expected_answer + solution_end
    )
    return [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": problem},
        {"role": "assistant", "content": final_prompt},
    ]

dataset["Messages"] = dataset.apply(format_dataset, axis=1)
dataset["N"] = dataset["Messages"].apply(
    lambda x: len(tokenizer.apply_chat_template(x))
)
dataset = dataset.loc[dataset["N"] <= max_seq_length // 2].copy()

from datasets import Dataset
dataset["text"] = tokenizer.apply_chat_template(
    dataset["Messages"].values.tolist(), tokenize=False
)
dataset = Dataset.from_pandas(dataset)

from trl import SFTTrainer, SFTConfig
trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    args=SFTConfig(
        dataset_text_field="text",
        per_device_train_batch_size=1,
        gradient_accumulation_steps=1,
        warmup_steps=5,
        num_train_epochs=2,
        learning_rate=2e-4,
        logging_steps=5,
        optim="adamw_8bit",
        weight_decay=0.001,
        lr_scheduler_type="linear",
        seed=3407,
        report_to="none",
    ),
)
trainer.train()

del dataset
torch.cuda.empty_cache()
import gc; gc.collect()


## Load GRPO Dataset


In [ ]:
dataset = load_dataset("open-r1/DAPO-Math-17k-Processed", "en", split="train")

def extract_hash_answer(text): return text

dataset = dataset.map(lambda x: {
    "prompt": [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": x["prompt"]},
    ],
    "answer": extract_hash_answer(x["solution"]),
})

import re
solution_end_regex = r"</SOLUTION>[\s]{0,}" + "(?:" + re.escape(tokenizer.eos_token) + ")?"
match_format = re.compile(
    rf"{reasoning_end}.*?"
    rf"{solution_start}(.+?){solution_end_regex}"
    rf"[\s]{{0,}}$",
    flags=re.MULTILINE | re.DOTALL
)

tokenized = dataset.map(
    lambda x: {"tokens": tokenizer.apply_chat_template(
        x["prompt"], add_generation_prompt=True, tokenize=True
    )},
    batched=True,
)
tokenized = tokenized.map(lambda x: {"L": len(x["tokens"])})
import numpy as np
maximum_length = int(np.quantile(tokenized["L"], 0.9))
dataset = dataset.select(np.where(np.array(tokenized["L"]) <= maximum_length)[0])
del tokenized
max_prompt_length = maximum_length + 1
max_completion_length = max_seq_length - max_prompt_length


In [ ]:
# ── AWPO reward: outcome (binary) ─────────────────────────────
def awpo_outcome_reward(completions, ground_truth=None, **kwargs):
    """Weighted outcome reward: func_selection + args_accuracy + execution."""
    if ground_truth is None:
        ground_truth = kwargs.get("ground_truths", [{}] * len(completions))
    scores = []
    for c, gt in zip(completions, ground_truth):
        response = c[0]["content"] if isinstance(c, list) else c
        if not isinstance(gt, dict):
            scores.append(0.0)
            continue
        expected_func = gt.get("function", "")
        expected_args = gt.get("arguments", {})
        call_match = match_format.search(response)
        if call_match is None:
            scores.append(0.0)
            continue
        try:
            import json
            call = json.loads("{" + call_match.group(1) + "}")
            fn_ok = 1.0 if call.get("function") == expected_func else 0.0
            actual_args = call.get("arguments", {})
            if expected_args:
                correct = sum(1 for k, v in expected_args.items()
                              if str(actual_args.get(k, "")).strip() == str(v).strip())
                args_ok = correct / len(expected_args)
            else:
                args_ok = 1.0
            score = 0.4 * fn_ok + 0.3 * args_ok + 0.3 * (1.0 if fn_ok and args_ok >= 0.8 else 0.0)
            scores.append(score)
        except:
            scores.append(0.0)
    return scores


In [ ]:
# ── AWPO reasoning quality reward ─────────────────────────────
import re

def reasoning_quality_reward(completions, **kwargs):
    """Reward reasoning structure: length + step markers."""
    scores = []
    for completion in completions:
        response = completion[0]["content"] if isinstance(completion, list) else completion
        reason_match = re.search(r"<start_working_out>(.*?)<end_working_out>", response, re.DOTALL)
        if not reason_match:
            scores.append(0.0)
            continue
        text = reason_match.group(1).strip()
        words = len(text.split())
        length_score = min(1.0, words / 50.0)
        has_steps = bool(re.search(
            r"(step\s*\d|first|then|finally|because|therefore|since)", text, re.I
        ))
        scores.append(0.7 * length_score + 0.3 * float(has_steps))
    return scores


In [ ]:
# ── AWPO advantage computation ────────────────────────────────
import math

def compute_group_stats(rewards):
    n = len(rewards)
    mean = sum(rewards) / n
    var = sum((r - mean) ** 2 for r in rewards) / n
    return {"mean": mean, "var": var, "std": math.sqrt(var)}

def variance_gate(sigma2_out, tau=0.5):
    """Hard binary gate: g=0 if variance>tau (outcome informative), else g=1."""
    return 0.0 if sigma2_out > tau else 1.0

def difficulty_weight(mu_out):
    """Inverted-U weight: w=4*mu*(1-mu), peaks at mu=0.5."""
    return 4.0 * mu_out * (1.0 - mu_out)

def awpo_group_advantages(outcome_rewards, reasoning_rewards, tau=0.5, eps=1e-8):
    n = len(outcome_rewards)
    out_stats = compute_group_stats(outcome_rewards)
    mu_out = out_stats["mean"]
    sigma2_out = out_stats["var"]
    g = variance_gate(sigma2_out, tau)
    w = difficulty_weight(mu_out)
    mixed = [(1.0 - g) * r_out + g * r_rea for r_out, r_rea in zip(outcome_rewards, reasoning_rewards)]
    mix_stats = compute_group_stats(mixed)
    mu_mix = mix_stats["mean"]
    std_mix = mix_stats["std"]
    advantages = [w * (r - mu_mix) / (std_mix + eps) for r in mixed]
    adaptive_clip_delta = g * 0.2
    return advantages, adaptive_clip_delta


In [ ]:
# ── AWPOTrainer ───────────────────────────────────────────────
import torch
from trl import GRPOConfig, GRPOTrainer
from vllm import SamplingParams
from typing import Any

class AWPOTrainer(GRPOTrainer):
    def __init__(self, *args, tau=0.5, adaptive_clip_delta=0.2, **kwargs):
        super().__init__(*args, **kwargs)
        self.tau = tau
        self.adaptive_clip_delta = adaptive_clip_delta
        self._reasoning_cache = {}

    def compute_advantages(self, rewards, group_indices, completions=None):
        B = rewards.shape[0]
        advantages = torch.zeros(B, dtype=torch.float32)
        clip_deltas = []
        unique_groups = group_indices.unique().tolist()
        for gid in unique_groups:
            mask = (group_indices == gid).nonzero(as_tuple=True)[0]
            g_idx = mask.tolist()
            out_r = [float(rewards[i]) for i in g_idx]
            if completions is not None:
                reason_r = [self._reasoning_cache.get(completions[i], 0.0) for i in g_idx]
            else:
                reason_r = [0.0] * len(g_idx)
            group_advs, clip_delta = awpo_group_advantages(out_r, reason_r, tau=self.tau)
            for local_idx, global_idx in enumerate(g_idx):
                advantages[global_idx] = group_advs[local_idx]
            clip_deltas.append(clip_delta)
        mean_clip_delta = sum(clip_deltas) / max(len(clip_deltas), 1)
        return advantages, mean_clip_delta

    def _cache_reasoning_rewards(self, completions):
        for c in completions:
            if c not in self._reasoning_cache:
                response = c[0]["content"] if isinstance(c, list) else c
                self._reasoning_cache[c] = 0.0

    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        clip_delta = getattr(self, "_last_clip_delta", 0.0)
        if clip_delta > 0.0:
            original_eps = self.args.epsilon
            self.args.epsilon = original_eps * (1.0 + clip_delta)
        loss_output = super().compute_loss(model, inputs, return_outputs, num_items_in_batch)
        if clip_delta > 0.0:
            self.args.epsilon = original_eps
        return loss_output


In [ ]:
# ── AWPO reward wrapper ───────────────────────────────────────
def awpo_reward_func(completions, ground_truth=None, **kwargs):
    if ground_truth is None:
        ground_truth = kwargs.get("ground_truths", [{}] * len(completions))
    return awpo_outcome_reward(completions, ground_truth)


In [ ]:
# ── Train AWPO ────────────────────────────────────────────────
vllm_sampling_params = SamplingParams(
    min_p=0.1, top_p=1.0, top_k=-1, seed=3407,
    stop=[tokenizer.eos_token], include_stop_str_in_output=True,
)
training_args = GRPOConfig(
    vllm_sampling_params=vllm_sampling_params,
    temperature=1.0, learning_rate=5e-6, weight_decay=0.001,
    warmup_ratio=0.1, lr_scheduler_type="linear", optim="adamw_8bit",
    logging_steps=1, per_device_train_batch_size=1,
    gradient_accumulation_steps=1, num_generations=8,
    max_prompt_length=max_prompt_length,
    max_completion_length=max_completion_length,
    max_steps=100, save_steps=100, report_to="none",
    output_dir="outputs/awpo_model",
)

trainer = AWPOTrainer(
    model=model,
    processing_class=tokenizer,
    reward_funcs=[awpo_reward_func, reasoning_quality_reward],
    args=training_args,
    train_dataset=dataset,
    tau=0.5,
)
trainer.train()


In [ ]:
# ── Inference ─────────────────────────────────────────────────
model.save_lora("awpo_saved_lora")

messages = [
    {"role": "system", "content": system_prompt},
    {"role": "user", "content": "What is the sqrt of 101?"},
]
text = tokenizer.apply_chat_template(
    messages, add_generation_prompt=True, tokenize=False,
)
sampling_params = SamplingParams(temperature=1.0, top_k=50, max_tokens=2048)
output = model.fast_generate(
    text,
    sampling_params=sampling_params,
    lora_request=model.load_lora("awpo_saved_lora"),
)[0].outputs[0].text
print(output)
